# SmolLM2 Integrated Memory V3 — does the complete model improve?

**GPU runtime → Run all.** Use `smoke` first to check the pipeline, then `balanced` for the research run.

This uses **HuggingFaceTB/SmolLM2-135M**, the official small SmolLM2 checkpoint and the same model as your uploaded results. The official family has 135M, 360M and 1.7B variants, rather than a 100M checkpoint. [Model card](https://huggingface.co/HuggingFaceTB/SmolLM2-135M).

Your V2 joint replacement of layers 0,18,29 gave these results:

| Context | PPL increase vs original | Full-model forward speedup |
|---|---:|---:|
| 256 | +0.19% | 0.67× |
| 512 | +0.98% | 0.96× |
| 1024 | +2.50% | 1.01× |

The partition mixer is promising for quality, especially layer 0, but V2 did not establish an overall win. Its T4 run used BF16; this notebook selects FP16 on T4 and native BF16 on Ampere/newer GPUs. Speed results need to be remeasured after that correction.

V3 trains the replacements **together**, uses multiple training contexts, and measures **complete-model cached decoding and total cache bytes**. The remaining Transformer layers stay active. No pretrained GPU result is pre-filled.

## Configurations and training

| Configuration | Changed layers | Purpose |
|---|---|---|
| Original | None | Unchanged pretrained model |
| Attention conservative | 0,29 | Matched jointly trained full-attention readouts |
| Attention expanded | 0,18,29 | Matched jointly trained full-attention readouts |
| Partition conservative | 0,29 | Keep sensitive layer 18 as original attention |
| Partition expanded | 0,18,29 | Test whether joint training recovers the V2 regression |
| Sink conservative | 0,29 | Untrained sink/local control |

Each learned model starts with training-only ridge calibration and attention transfer. It then trains all replacement cores jointly on actual student states, minimizing

$$L=\operatorname{CE}(y,p_s)+\lambda\tau^2\operatorname{KL}(p_t^{(\tau)}\Vert p_s^{(\tau)}).$$

The original embeddings, FFNs, Q/K/V/O projections and norms remain frozen. Readout calibration is folded into O for inference. Training context rotates through 256,512,1024 in `balanced`; selection minimizes average validation NLL across those contexts. Test data is accessed only after selection is locked.

This retrains the **architectures**, because the uploaded result tables do not include the learned core checkpoints. It does not pretend to reload weights from a CSV. Best adapter weights and exact model revision are exported for subsequent use.

In [ ]:
import os, sys, json, subprocess, tempfile, shutil
from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

REPO_REF = "main"
REPO = Path(tempfile.mkdtemp(prefix="smollm2-integrated-v3-")) / "TinyCeNN-LM"
subprocess.run(["git", "clone", "--quiet", "https://github.com/vtavakkoli/TinyCeNN-LM.git", str(REPO)], check=True)
subprocess.run(["git", "checkout", REPO_REF], cwd=REPO, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "transformers==4.57.6", "datasets>=3,<5",
                "pytest", "nbformat", "pandas", "matplotlib"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(REPO), "--no-deps"], check=True)
os.environ["PYTHONPATH"] = os.pathsep.join([str(REPO), str(REPO / "src")])
sys.path[:0] = [str(REPO), str(REPO / "src")]
import torch
print("Source:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## Choose budget and save location

`balanced` runs five configurations plus the original baseline. Joint training and cached full-model evaluation are more expensive than the V2 layer screen. `extended` increases data, updates and context lengths substantially. No runtime guarantee is implied. Drive saves occur during training; run directories are unique and are never overwritten. Interrupted experiments retain best checkpoints, but optimizer-state resume is not implemented.

In [ ]:
PROFILE = "balanced" # @param ["smoke", "balanced", "extended"]
SAVE_TO_DRIVE = True # @param {type:"boolean"}
SEED = 2028 # @param {type:"integer"}
PROFILES = {
    "smoke": dict(train_contexts="32,64", test_contexts="32,64,128", block_size=8, features=16,
                  train_documents=4, validation_documents=2, test_documents=2, warm_documents=2,
                  warm_steps=2, joint_steps=4, eval_every=2, timing_documents=1, timing_repeats=1, decode_tokens=8),
    "balanced": dict(train_contexts="256,512,1024", test_contexts="256,512,1024,2048", block_size=32, features=64,
                     train_documents=128, validation_documents=16, test_documents=32, warm_documents=16,
                     warm_steps=100, joint_steps=300, eval_every=50, timing_documents=3, timing_repeats=3, decode_tokens=32),
    "extended": dict(train_contexts="512,1024,2048", test_contexts="512,1024,2048,4096", block_size=32, features=64,
                     train_documents=256, validation_documents=32, test_documents=64, warm_documents=32,
                     warm_steps=200, joint_steps=1000, eval_every=100, timing_documents=5, timing_repeats=5, decode_tokens=64),
}
if not torch.cuda.is_available() and PROFILE != "smoke":
    raise RuntimeError("Select a GPU runtime, or smoke for a CPU pipeline check.")
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/TinyCeNN/smollm2-integrated-v3")
else:
    BASE = Path("/content/smollm2-integrated-v3-results") if Path("/content").exists() else Path.cwd()/"v3-results"
BASE.mkdir(parents=True, exist_ok=True)
run_id = PROFILE + "-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
OUT, LOG = BASE / run_id, BASE / (run_id + ".log")
config = dict(PROFILES[PROFILE], seed=SEED)
print(json.dumps(config, indent=2))
print("Results:", OUT)

## Fresh data and optional additional exclusions

The repository includes the **144 document hashes from your supplied V2 report**, which are excluded from all V3 partitions. V1/V2 holdout buckets 0..39 are also excluded. V3 uses test buckets 40..49, validation 50..59 and training 60..99. Other past experiments may have used these buckets for training: provide their manifests too if needed. This cannot rule out the pretrained model having seen web documents during its original training.

No previous token blocks or core weights are needed for this run.

In [ ]:
ADDITIONAL_MANIFESTS = []  # Optional Drive paths to earlier manifest.json files.
UPLOAD_MANIFESTS = False # @param {type:"boolean"}
if UPLOAD_MANIFESTS:
    from google.colab import files
    folder = Path(tempfile.mkdtemp(prefix="v3-exclusions-"))
    for index, (name, content) in enumerate(files.upload().items()):
        value = json.loads(content)
        if not value.get("document_hashes"):
            raise ValueError(f"{name}: expected a manifest with document_hashes")
        path = folder / f"manifest-{index}.json"
        path.write_bytes(content)
        ADDITIONAL_MANIFESTS.append(str(path))
print("Additional manifests:", len(ADDITIONAL_MANIFESTS))

## CPU preflight, then training

Preflight uses random tiny Llama models and no downloads. It checks joint gradients, frozen teacher weights, checkpoint reload, native T4 precision choice, and full-sequence versus cached logits for every layer position and mixer. The complete offline experiment also executes this notebook's results cell. Passing preflight establishes correctness for those cases, not pretrained-model quality.

In [ ]:
env = dict(os.environ, CUDA_VISIBLE_DEVICES="", OMP_NUM_THREADS="1", MKL_NUM_THREADS="1")
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_integrated_memory.py"],
               cwd=REPO, env=env, check=True)

In [ ]:
command = [sys.executable, "-u", str(REPO / "scripts/benchmark_smollm2_integrated_memory.py"),
           "--output-dir", str(OUT)]
for key, value in config.items():
    command += ["--" + key.replace("_", "-"), str(value)]
for path in ADDITIONAL_MANIFESTS:
    command += ["--exclude-manifest", str(path)]
print(" ".join(command))
try:
    with LOG.open("w") as log:
        with subprocess.Popen(command, cwd=REPO, env=os.environ.copy(), stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
                log.flush()
            status = process.wait()
    if status:
        raise RuntimeError(f"Run failed with code {status}; inspect {LOG}")
except BaseException as error:
    OUT.mkdir(parents=True, exist_ok=True)
    (OUT / "failure_report.json").write_text(json.dumps({"error":str(error),"log":str(LOG)},indent=2))
    raise
finally:
    if OUT.exists() and LOG.exists():
        shutil.copy2(LOG, OUT / "console.log")
        archive = shutil.make_archive(str(OUT)+"-results", "zip", root_dir=OUT)
        print("Archive:", archive)

## Is the complete model better?

The table compares each model with the original Transformer, the **matched adapted attention control**, and its **own pre-joint-training checkpoint**. Lower perplexity ratios are better. For prefill and decoding, speedup above 1 is better. Cache ratio includes the caches in all remaining Transformer layers.

A quality-win flag requires upper paired 95% NLL bounds below zero against both Transformer references. A quality-preserving efficiency flag allows at most 0.02 nats/token regression against both and requires faster complete-model prefill **and** decode plus less total cache. Flags require at least eight test documents. Timing has no statistical win guarantee; inspect repeated samples and repeat promising runs with additional seeds.

The benchmark uses batch-one causal cached decoding with identical input tokens across models, full projections, and only the last-position LM-head output. It excludes loading and warmup. Greedy examples are shown separately and are not used for architecture selection.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

report = json.loads((OUT / "integrated_report.json").read_text())
assert report["status"] == "completed", "Run is incomplete"
results = pd.DataFrame(report["rows"])
compared = results.loc[results.matched_control.notna()].copy()
columns = ["candidate","context","ppl_ratio","adapted_ppl_ratio","before_joint_ppl_ratio",
           "prefill_speedup","decode_speedup","total_cache_ratio","beats_both_quality",
           "quality_preserving_efficiency_win","selected_on_validation"]
display(compared[columns].round(5))
compared[columns].to_csv(OUT / "decision_table.csv", index=False)
print("Locked validation selection:", report["selected"])
print("Native dtype:", report["native_dtype"], "GPU:", report["gpu"])

for context, part in compared.groupby("context"):
    part = part.sort_values("candidate")
    y = np.arange(len(part))
    fig, axes = plt.subplots(1,4,figsize=(19,4.4),sharey=True)
    labels = part.candidate.tolist()
    for ax, prefix, title in [(axes[0],"","PPL / original"),(axes[1],"adapted_","PPL / adapted control")]:
        lo,hi = np.exp(part[prefix+"ci_low"]),np.exp(part[prefix+"ci_high"])
        ax.hlines(y,lo,hi,color="#247ba0")
        ax.scatter(part[prefix+"ppl_ratio"],y,color="#247ba0")
        ax.axvline(1,color="black",linestyle="--")
        ax.set_title(title+"\nLower is better; paired 95% intervals")
    axes[2].scatter(part.prefill_speedup,y-.1,label="Full-model prefill")
    axes[2].scatter(part.decode_speedup,y+.1,label="Cached decode",marker="s")
    axes[2].axvline(1,color="black",linestyle="--")
    axes[2].set_xscale("log")
    axes[2].set_title("Original time / candidate time\nHigher is better")
    axes[2].legend(fontsize=8)
    axes[3].scatter(part.total_cache_ratio,y,color="#008577")
    axes[3].axvline(1,color="black",linestyle="--")
    axes[3].set_xlim(left=0)
    axes[3].set_title("Total cache / original total cache\nLower is better")
    axes[0].set_yticks(y,labels)
    axes[0].invert_yaxis()
    for ax in axes: ax.grid(axis="x",alpha=.18)
    fig.suptitle(f"{report['args']['base_model']} · integrated V3 · T{context} · "
                 f"seed {report['args']['seed']} · {report['args']['test_documents']} test documents")
    fig.tight_layout()
    for ext in ("png","svg"):
        fig.savefig(OUT/f"integrated-T{context}.{ext}",dpi=150,bbox_inches="tight")
    plt.show()
    plt.close(fig)

history = pd.read_csv(OUT / "training_history.csv")
fig,ax = plt.subplots(figsize=(10,4))
for name,part in history.groupby("candidate"):
    ax.plot(part.step,part.validation_nll,marker="o",label=name)
ax.set(xlabel="Joint update",ylabel="Mean validation NLL",title="Multi-context validation during joint training")
ax.legend(fontsize=8)
ax.grid(alpha=.2)
fig.tight_layout()
fig.savefig(OUT/"joint-training.png",dpi=150)
plt.show()
plt.close(fig)
examples = json.loads((OUT / "generation_examples.json").read_text())
for example in examples:
    if example["candidate"] in ("original",report["selected"]) and example["prompt_index"]==0:
        print("\n", example["candidate"], "\nPrompt:", example["prompt"], "\nContinuation:", example["continuation"])


## Optional: reload the selected model and try a prompt

The export contains adapter weights rather than a second copy of the entire base model. Reload uses the **exact saved base-model revision**, then installs the selected adapters. SmolLM2-135M is a base completion model, so these are continuations rather than instruction-tuned chat answers. Use the supplied greedy decoder for mixed caches; beam search and cache cropping are unsupported.

In [ ]:
TRY_PROMPT = False # @param {type:"boolean"}
PROMPT = "The purpose of scientific research is" # @param {type:"string"}
if TRY_PROMPT:
    from transformers import AutoTokenizer, AutoModelForCausalLM
    from tinycenn_lm.integrated_memory import native_dtype, restore_student, inference_mode, greedy_generate
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    dtype = native_dtype(device)
    teacher = AutoModelForCausalLM.from_pretrained(report["args"]["base_model"],
        revision=report["model_revision"], torch_dtype=dtype, attn_implementation="sdpa").to(device).eval()
    tokenizer = AutoTokenizer.from_pretrained(report["args"]["base_model"],revision=report["model_revision"])
    record = next(r for r in report["candidates"] if r["candidate"]==report["selected"])
    payload = torch.load(OUT/record["checkpoint"],map_location="cpu",weights_only=True)
    assert payload["metadata"]["model_revision"] == report["model_revision"]
    student = restore_student(teacher,payload)
    ids = tokenizer(PROMPT,return_tensors="pt").input_ids.to(device)
    for name,model in [("Original",teacher),(report["selected"],student)]:
        with inference_mode(model,str(dtype).removeprefix("torch.")):
            tokens,_ = greedy_generate(model,ids,tokens=64)
        print(name+":",tokenizer.decode(tokens[0],skip_special_tokens=True))
    del student,teacher

## Download the results

Share the ZIP for review. It contains the model/data/source revisions, fresh data hashes, adapter checkpoints before and after joint training, validation selection, per-document test NLL, full-model timing samples, generated examples, tables and plots. A completed run can still be worse than the Transformer; use the decision table.

In [ ]:
archive = shutil.make_archive(str(OUT)+"-results", "zip", root_dir=OUT)
print("Results:",archive)
DOWNLOAD_ZIP = True # @param {type:"boolean"}
if DOWNLOAD_ZIP:
    from google.colab import files
    files.download(archive)